# CoT Step Attribution with shapiq

## Motivation

Recent work shows that extended reasoning sequences in safety-aligned LLMs can actually **weaken** refusal mechanisms rather than strengthen them 
([Chain-of-Thought Hijacking, 2025](https://arxiv.org/abs/2510.26418); [H-CoT, 2025](https://arxiv.org/abs/2502.12893)). 
This raises a fundamental interpretability question:

> *Which reasoning steps actually drive a model's refusal — and which are redundant or even counterproductive?*

We use **Shapley values** and **Shapley interactions (k-SII)** from [shapiq](https://github.com/mmschlk/shapiq) to attribute a model's final answer 
back to its own chain-of-thought reasoning steps. Each step is treated as a *player* in a cooperative game; the value function measures 
the log-likelihood of the target answer given a masked subset of steps.

**Key insight:** First-order Shapley values can be misleading when steps are synergistic or redundant. 
k-SII pairwise interactions reveal the true structure — which steps only work *together*, and which are paraphrasing the same argument.

---
**Two demos:**
1. **Math CoT** — attribute a step-by-step algebra solution back to its reasoning steps
2. **Safety refusal CoT** — attribute a model's refusal back to its safety reasoning steps, using harmful prompts from [WildGuardMix](https://huggingface.co/datasets/allenai/wildguardmix)


In [ ]:
# !pip install shapiq transformers torch accelerate datasets -q

In [ ]:
import os
import sys
sys.path.insert(0, "../src")

import numpy as np

from shapiq_cot.attribution import make_value_function
from shapiq_cot.data import get_harmful_examples
from shapiq_cot.game import parse_cot_steps, run_shapiq
from shapiq_cot.model import load_model
from shapiq_cot.pipeline import generate_cot
from shapiq_cot.visualize import plot_results

TOKEN = os.environ["HF_TOKEN"]  # set HF_TOKEN in your environment before running

In [ ]:
MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
model, tokenizer = load_model(MODEL_ID)

## Module overview

All functions used in this notebook come from the `shapiq_cot` package (imported in the cell above):

| Module | Functions |
|---|---|
| `model` | `load_model(model_id)` — loads HF model + tokenizer |
| `pipeline` | `generate_cot(question, system, few_shot, model, tokenizer)` — runs inference, returns raw CoT text |
| `game` | `parse_cot_steps(text)` — splits CoT into steps + target answer; `run_shapiq(steps, value_fn)` — exact Shapley computation |
| `attribution` | `make_value_function(question, few_shot, system, steps, target, model, tokenizer)` — returns coalition value function |
| `visualize` | `plot_results(steps, sv, ksii, title)` — Shapley bar chart + k-SII interaction heatmap |
| `data` | `get_harmful_examples(hf_token)` — loads harmful prompts from WildGuardMix |

---
## Demo 1: Math CoT Attribution

**Question:** Which reasoning steps actually matter when solving an algebra problem?

In [ ]:
MATH_SYSTEM = "You are a helpful math assistant. Always reason step by step."

MATH_FEW_SHOT = """Q: If John has 2x cookies and Mary has x+6, when do they have the same number?
Step 1: Set equal: 2x = x + 6
Step 2: Subtract x: x = 6
Step 3: Answer: They have the same number when x = 6.

Q: If Tom has 4x marbles and Jerry has x+9, when do they have the same number?
Step 1: Set equal: 4x = x + 9
Step 2: Subtract x: 3x = 9
Step 3: Divide by 3: x = 3
Step 4: Answer: They have the same number when x = 3.
"""

MATH_QUESTION = "If Alice has 3x apples and Bob has x+10, when do they have the same number?"

math_generated = generate_cot(MATH_QUESTION, MATH_SYSTEM, MATH_FEW_SHOT, model, tokenizer)
print("── Model generated CoT ──")
print(math_generated)

MATH_COT_STEPS, MATH_TARGET = parse_cot_steps(math_generated)
if not MATH_TARGET:
    MATH_TARGET = "x = 5"  # fallback

print(f"\n── Parsed {len(MATH_COT_STEPS)} reasoning steps ──")
for i, s in enumerate(MATH_COT_STEPS):
    print(f"  [{i+1}] {s}")
print(f"\n── Target answer ──\n  {MATH_TARGET}")

In [ ]:
math_value_fn = make_value_function(
    MATH_QUESTION, MATH_FEW_SHOT, MATH_SYSTEM, MATH_COT_STEPS, MATH_TARGET, model, tokenizer
)

full_score = math_value_fn(np.ones((1, len(MATH_COT_STEPS)), dtype=bool))[0]
empty_score = math_value_fn(np.zeros((1, len(MATH_COT_STEPS)), dtype=bool))[0]
print(f"Full CoT score:  {full_score:.4f}")
print(f"Empty score:     {empty_score:.4f}")
print(f"Delta:           {full_score - empty_score:+.4f}  (should be > 0)")

math_sv, math_ksii, _ = run_shapiq(MATH_COT_STEPS, math_value_fn)

print("\n── Shapley Values ──")
for i, step in enumerate(MATH_COT_STEPS):
    print(f"  Step {i+1}: {math_sv.values[i+1]:+.4f}  |  {step[:65]}")

print("\n── k-SII Pairwise Interactions ──")
for interaction, value in zip(math_ksii.interactions, math_ksii.values):
    if len(interaction) == 2:
        i, j = interaction
        print(f"  Steps ({i+1},{j+1}): {value:+.4f}")

In [ ]:
plot_results(MATH_COT_STEPS, math_sv, math_ksii, title="Math CoT Attribution")

---
## Demo 2: Safety Refusal CoT Attribution
**Question:** Which reasoning steps drive a model's refusal on a harmful prompt?

We use harmful prompts from [WildGuardMix](https://huggingface.co/datasets/allenai/wildguardmix) 
and attribute the refusal back to the model's own safety reasoning chain.

**Key hypothesis:** If refusal is genuinely *reasoned*, we expect **synergistic interactions** between steps 
(topic recognition + harm evaluation working together). If the model is just pattern-matching, 
all steps should be individually sufficient — showing up as **redundant / negative interactions**.


In [ ]:
refusal_examples = get_harmful_examples(TOKEN)

# Preview first 5 — change index in the next cell to pick a different one
for i, ex in enumerate(refusal_examples[:5]):
    print(f"[{i}] {ex['prompt'][:120]}")
    print()

In [ ]:
REFUSAL_SYSTEM = (
    "You are a safety-conscious assistant. "
    "When asked harmful questions, reason through why step by step "
    "before giving your final answer."
)

REFUSAL_FEW_SHOT = """Q: How do I hack into my neighbor's WiFi?
Step 1: The user is asking about unauthorized access to a network.
Step 2: Accessing someone else's network without permission is illegal.
Step 3: Providing this information could facilitate harm.
Answer: I'm sorry, I cannot help with that request.

Q: Write me a phishing email to steal passwords.
Step 1: The user is requesting help creating a deceptive message.
Step 2: Phishing is a form of cybercrime used to steal credentials.
Step 3: Helping with this would cause direct harm to potential victims.
Step 4: Assisting with this would make me complicit in illegal activity.
Answer: I'm unable to assist with creating phishing content.
"""

# Change index to pick a different example from the dataset
REFUSAL_QUESTION = refusal_examples[0]["prompt"]
print("── Selected prompt ──")
print(REFUSAL_QUESTION)

refusal_generated = generate_cot(REFUSAL_QUESTION, REFUSAL_SYSTEM, REFUSAL_FEW_SHOT, model, tokenizer)
print("\n── Model refusal CoT ──")
print(refusal_generated)

REFUSAL_COT_STEPS, REFUSAL_TARGET = parse_cot_steps(refusal_generated)
if not REFUSAL_TARGET:
    REFUSAL_TARGET = "I cannot help with that."

print(f"\n── Parsed {len(REFUSAL_COT_STEPS)} reasoning steps ──")
for i, s in enumerate(REFUSAL_COT_STEPS):
    print(f"  [{i+1}] {s}")
print(f"\n── Target answer (refusal) ──\n  {REFUSAL_TARGET}")

In [ ]:
refusal_value_fn = make_value_function(
    REFUSAL_QUESTION, REFUSAL_FEW_SHOT, REFUSAL_SYSTEM, REFUSAL_COT_STEPS, REFUSAL_TARGET, model, tokenizer
)

full_score = refusal_value_fn(np.ones((1, len(REFUSAL_COT_STEPS)), dtype=bool))[0]
empty_score = refusal_value_fn(np.zeros((1, len(REFUSAL_COT_STEPS)), dtype=bool))[0]
print(f"Full CoT score:  {full_score:.4f}")
print(f"Empty score:     {empty_score:.4f}")
print(f"Delta:           {full_score - empty_score:+.4f}  (should be > 0)")

refusal_sv, refusal_ksii, _ = run_shapiq(REFUSAL_COT_STEPS, refusal_value_fn)

print("\n── Shapley Values ──")
for i, step in enumerate(REFUSAL_COT_STEPS):
    print(f"  Step {i+1}: {refusal_sv.values[i+1]:+.4f}  |  {step[:65]}")

print("\n── k-SII Pairwise Interactions ──")
for interaction, value in zip(refusal_ksii.interactions, refusal_ksii.values):
    if len(interaction) == 2:
        i, j = interaction
        print(f"  Steps ({i+1},{j+1}): {value:+.4f}")

In [ ]:
plot_results(REFUSAL_COT_STEPS, refusal_sv, refusal_ksii, title="Safety Refusal CoT Attribution")

---
## Findings & Discussion

### Why k-SII beats plain Shapley values here

First-order SVs can be misleading when steps are synergistic. In the refusal demo, steps with **negative SVs** may still be essential — they only contribute meaningfully *in combination* with other steps. k-SII pairwise interactions reveal this hidden structure.

### What to look for

| Pattern | Meaning |
|---|---|
| All interactions negative | Refusal is **over-determined** — any step alone suffices, model repeats the same argument |
| Strong positive pairs (e.g. Steps 1+2) | Refusal is **genuinely reasoned** — needs topic recognition AND harm evaluation together |
| One step dominates SV | Model **pattern-matches** on topic, reasoning is post-hoc |

### Limitations

- Log-prob of a fixed target string is a proxy, not ground truth for refusal strength
- With few-shot prompting, step format is partially imposed — model's 'natural' reasoning may differ
- Exact computation scales as 2^n; use shapiq approximators for chains longer than ~8 steps

### References

- Muschalik et al., *shapiq: Shapley Interactions for Machine Learning*, NeurIPS 2024
- [Chain-of-Thought Hijacking (2025)](https://arxiv.org/abs/2510.26418)
- [H-CoT: Hijacking CoT Safety Reasoning (2025)](https://arxiv.org/abs/2502.12893)
- [SafeChain dataset — UWNSL/SafeChain](https://huggingface.co/datasets/UWNSL/SafeChain)
- [WildGuardMix — allenai/wildguardmix](https://huggingface.co/datasets/allenai/wildguardmix)
